# Ablation Tests

Compare two completed `ablation_analysis.ipynb` runs, typically a depth-2 model and a depth-4 model. The notebook plots center-resolved ablation heatmaps for `d_pert = 1` and `d_pert = 2` for both runs, the absolute difference, and the relative difference. It also overlays population-level ablation curves for the two runs.


**Setup**

In [ ]:
from pathlib import Path
import json
import re
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "legend.frameon": False,
})

print("PROJECT_ROOT =", PROJECT_ROOT)


**Settings**

Set `DEPTH2_RUN_DIR` and `DEPTH4_RUN_DIR` to completed `ablation_analysis.ipynb` result folders. If either is `None`, the notebook tries to discover the newest complete run with the matching model depth.


In [ ]:
EXPERIMENT_GROUP = "ablation_tests"
ABLATION_RESULTS_ROOT = PROJECT_ROOT / "results_experiments" / "ablation_analysis"

DEPTH2_RUN_DIR = PROJECT_ROOT / "results_experiments/ablation_analysis/rest_folded_depth2_after_cleanup_20260724_155016" #rest_folded_depth2_after_cleanup_20260709_170140"
DEPTH4_RUN_DIR = PROJECT_ROOT / "results_experiments/ablation_analysis/rest_folded_depth4_after_cleanup_20260724_142135"

DEPTH2_LABEL = "depth 2"
DEPTH4_LABEL = "depth 4"

METRIC = "delta_mu"  # "delta_mu", "delta_variance", or "delta_mse"
HOPS_TO_PLOT = [1, 2]
MIN_CASES = None  # Set to None to use all finite cells.

# Relative difference is 100 * (depth4 - depth2) / abs(depth2).
RELATIVE_DIFFERENCE_EPS = 1e-12
RELATIVE_DIFFERENCE_CLIP_PERCENTILE = 98

# Population curves: plot both conditional and prevalence-weighted estimands.
POPULATION_ESTIMANDS = {
    "conditional": {
        "value": "conditional_effect",
        "sem": "conditional_sem",
        "label": "conditional effect",
    },
    "prevalence_weighted": {
        "value": "prevalence_weighted_effect",
        "sem": "prevalence_weighted_sem",
        "label": "prevalence-weighted effect",
    },
}
POPULATION_HOPS_TO_PLOT = None  # None uses all available hops.
POPULATION_N_COLS = 4
RUN_COLORS = {
    DEPTH2_LABEL: "#2f6f9f",
    DEPTH4_LABEL: "#c65f32",
}

# Saved graph curvature targets use units of 1 / (10 um)^2.
GRAPH_CURVATURE_LENGTH_UNIT_UM = 10.0
CURVATURE_TO_PHYSICAL_FACTOR = 1.0 / (GRAPH_CURVATURE_LENGTH_UNIT_UM ** 2)
CURVATURE_SQUARED_TO_PHYSICAL_FACTOR = CURVATURE_TO_PHYSICAL_FACTOR ** 2
CURVATURE_UNIT_LABEL = r"$\mu m^{-2}$"
CURVATURE_SQUARED_UNIT_LABEL = r"$\mu m^{-4}$"

METRIC_SPECS = {
    "delta_mu": {
        "label": rf"$\Delta$ predicted curvature ({CURVATURE_UNIT_LABEL})",
        # Match plot_experiment_results.ipynb: positive means ablation decreases curvature.
        "scale": -CURVATURE_TO_PHYSICAL_FACTOR,
        "default_limits": (-6.5e-5, 6.5e-5),
        "default_ticks": [-6e-5, -3e-5, 0.0, 3e-5, 6e-5],
        "short": "curvature",
    },
    "delta_variance": {
        "label": rf"$\Delta$ predicted variance ({CURVATURE_SQUARED_UNIT_LABEL})",
        "scale": CURVATURE_SQUARED_TO_PHYSICAL_FACTOR,
        "default_limits": None,
        "default_ticks": None,
        "short": "variance",
    },
    "delta_mse": {
        "label": rf"$\Delta$ squared error ({CURVATURE_SQUARED_UNIT_LABEL})",
        "scale": CURVATURE_SQUARED_TO_PHYSICAL_FACTOR,
        "default_limits": None,
        "default_ticks": None,
        "short": "mse",
    },
}

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP / f"{METRIC}_{RUN_TIMESTAMP}"
FIGURES_DIR = SAVE_DIR / "figures"
TABLES_DIR = SAVE_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_FORMATS = ("svg", "png")
CMAP = "RdBu_r"
BAD_COLOR = "white"
DIFFERENCE_LABEL = f"{DEPTH4_LABEL} - {DEPTH2_LABEL}"


**Load Runs**

In [ ]:
REQUIRED_CENTER_TABLE = "center_resolved_effect_summary.csv"
REQUIRED_POPULATION_TABLE = "population_effect_summary.csv"
OPTIONAL_VARIANCE_TABLES = [
    "center_marker_pair_curvature_variance_summary_post_mad.csv",
    "center_marker_pair_curvature_variance_summary.csv",
    "center_marker_pair_curvature_variance_summary_post_non_overlap.csv",
]


def load_json(path):
    with Path(path).open() as handle:
        return json.load(handle)


def result_settings_path(run_dir):
    run_dir = Path(run_dir)
    for name in ("settings.json", "config.json"):
        path = run_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(f"No settings.json or config.json in {run_dir}")


def load_run(run_dir):
    run_dir = Path(run_dir).expanduser().resolve()
    settings = load_json(result_settings_path(run_dir))
    center_path = run_dir / "tables" / REQUIRED_CENTER_TABLE
    population_path = run_dir / "tables" / REQUIRED_POPULATION_TABLE
    if not center_path.exists():
        raise FileNotFoundError(center_path)
    if not population_path.exists():
        raise FileNotFoundError(population_path)
    center = pd.read_csv(center_path)
    population = pd.read_csv(population_path)
    variance = None
    variance_table_name = None
    for table_name in OPTIONAL_VARIANCE_TABLES:
        variance_path = run_dir / "tables" / table_name
        if variance_path.exists():
            variance = pd.read_csv(variance_path)
            variance_table_name = table_name
            break
    return {
        "run_dir": run_dir,
        "settings": settings,
        "summary": center,
        "population": population,
        "variance": variance,
        "variance_table_name": variance_table_name,
    }


def discover_latest_ablation_run(depth):
    candidates = []
    for run_dir in ABLATION_RESULTS_ROOT.iterdir():
        if not run_dir.is_dir():
            continue
        table_path = run_dir / "tables" / REQUIRED_CENTER_TABLE
        population_path = run_dir / "tables" / REQUIRED_POPULATION_TABLE
        if not table_path.exists() or not population_path.exists():
            continue
        try:
            settings = load_json(result_settings_path(run_dir))
        except Exception:
            continue
        model_depth = settings.get("model", {}).get("num_layers")
        if model_depth is None:
            match = re.search(r"depth(\d+)", run_dir.name)
            model_depth = int(match.group(1)) if match else None
        if int(model_depth) != int(depth):
            continue
        candidates.append(run_dir)
    if not candidates:
        raise FileNotFoundError(f"No complete ablation run found for depth={depth}.")
    return max(candidates, key=lambda path: path.stat().st_mtime)


depth2_run_dir = Path(DEPTH2_RUN_DIR) if DEPTH2_RUN_DIR is not None else discover_latest_ablation_run(2)
depth4_run_dir = Path(DEPTH4_RUN_DIR) if DEPTH4_RUN_DIR is not None else discover_latest_ablation_run(4)

runs = {
    DEPTH2_LABEL: load_run(depth2_run_dir),
    DEPTH4_LABEL: load_run(depth4_run_dir),
}

run_summary = pd.DataFrame([
    {
        "label": label,
        "run_dir": str(run["run_dir"]),
        "num_layers": run["settings"].get("model", {}).get("num_layers"),
        "analysis_mode": run["settings"].get("analysis_mode"),
        "n_folds": run["settings"].get("split", {}).get("n_folds"),
        "timepoint_filter_mode": run["settings"].get("filtering", {}).get("timepoint_filter_mode"),
    }
    for label, run in runs.items()
])
run_summary.to_csv(TABLES_DIR / "ablation_test_run_summary.csv", index=False)
display(run_summary)


**Prepare Heatmap Arrays**

In [ ]:
def marker_names_from_run(run):
    names = run["settings"].get("marker_names")
    if names is None:
        names = run["settings"].get("dataset", {}).get("marker_names")
    if names is None:
        table = run["summary"]
        names = sorted(table["center_marker"].dropna().unique().tolist())
    return list(names)


marker_names = marker_names_from_run(runs[DEPTH2_LABEL])
marker_names_4 = marker_names_from_run(runs[DEPTH4_LABEL])
if marker_names != marker_names_4:
    raise ValueError(f"Marker names differ between runs: {marker_names} vs {marker_names_4}")
marker_to_index = {marker: index for index, marker in enumerate(marker_names)}


def marker_value_to_index(value):
    if value in marker_to_index:
        return marker_to_index[value]
    try:
        index = int(float(value))
    except (TypeError, ValueError):
        return None
    return index if 0 <= index < len(marker_names) else None


def scaled_metric_table(run, metric):
    table = run["summary"].copy()
    table = table[table["metric"] == metric].copy()
    if table.empty:
        raise ValueError(f"No rows for metric {metric!r} in {run['run_dir']}")
    scale = METRIC_SPECS[metric]["scale"]
    for column in ("mean_effect", "std_effect"):
        if column in table.columns:
            table[column] = pd.to_numeric(table[column], errors="coerce") * scale
    return table


def heatmap_array(run, metric, hop):
    table = scaled_metric_table(run, metric)
    subset = table[table["hop"].astype(int) == int(hop)]
    n = len(marker_names)
    values = np.full((n, n), np.nan, dtype=float)
    counts = np.zeros((n, n), dtype=int)
    for row in subset.itertuples(index=False):
        center_index = marker_to_index.get(row.center_marker)
        source_index = marker_value_to_index(row.source_marker)
        if center_index is None or source_index is None:
            continue
        values[center_index, source_index] = float(row.mean_effect)
        counts[center_index, source_index] = int(row.cases)
    return values, counts


combined_tables = []
for label, run in runs.items():
    table = scaled_metric_table(run, METRIC)
    table.insert(0, "run_label", label)
    table.insert(1, "run_dir", str(run["run_dir"]))
    combined_tables.append(table)
combined_metric_df = pd.concat(combined_tables, ignore_index=True)
combined_metric_df.to_csv(TABLES_DIR / f"combined_{METRIC}_center_resolved_summary.csv", index=False)
combined_metric_df.head()


**Plot Helpers**

In [ ]:
def save_figure(fig, name):
    paths = []
    for suffix in FIGURE_FORMATS:
        path = FIGURES_DIR / f"{name}.{suffix}"
        fig.savefig(path, bbox_inches="tight")
        paths.append(path)
    print("Saved", paths[0])
    return paths


def cmap_with_bad(name, bad_color=BAD_COLOR):
    cmap = plt.get_cmap(name).copy()
    cmap.set_bad(bad_color)
    return cmap


def draw_heatmap(ax, values, *, vmin, vmax, cmap, low_count_mask=None):
    n_rows, n_cols = values.shape
    masked = np.ma.masked_invalid(values)
    image = ax.pcolormesh(
        np.arange(n_cols + 1),
        np.arange(n_rows + 1),
        masked,
        shading="flat",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        edgecolors="white",
        linewidth=0.6,
        antialiased=False,
        snap=True,
    )
    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)
    ax.set_aspect("equal")
    ax.grid(False)

    if low_count_mask is not None:
        for row_index, col_index in np.argwhere(low_count_mask):
            ax.add_patch(Rectangle(
                (col_index, row_index),
                1,
                1,
                facecolor="none",
                edgecolor="0.35",
                hatch="////",
                linewidth=0,
                zorder=5,
            ))
    return image


def format_heatmap_axis(ax, *, title, show_ylabel=False):
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xticks(np.arange(len(marker_names)) + 0.5)
    ax.set_xticklabels(marker_names, rotation=60, ha="right", fontsize=9)
    ax.set_yticks(np.arange(len(marker_names)) + 0.5)
    ax.set_yticklabels(marker_names if show_ylabel else [], fontsize=9)
    ax.set_xlabel("perturbation marker")
    if show_ylabel:
        ax.set_ylabel("center marker")


def symmetric_limits(*arrays, fallback=None):
    if fallback is not None:
        return fallback
    finite = np.concatenate([np.ravel(arr[np.isfinite(arr)]) for arr in arrays])
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return (-1.0, 1.0)
    lim = float(np.nanpercentile(np.abs(finite), 98))
    if not np.isfinite(lim) or lim <= 0:
        lim = float(np.nanmax(np.abs(finite)))
    if not np.isfinite(lim) or lim <= 0:
        lim = 1.0
    return (-lim, lim)


**Comparison Heatmaps**

In [ ]:
metric_spec = METRIC_SPECS[METRIC]
metric_limits = metric_spec.get("default_limits")
metric_ticks = metric_spec.get("default_ticks")
cmap = cmap_with_bad(CMAP)

for hop in HOPS_TO_PLOT:
    values2, counts2 = heatmap_array(runs[DEPTH2_LABEL], METRIC, hop)
    values4, counts4 = heatmap_array(runs[DEPTH4_LABEL], METRIC, hop)

    valid2 = np.isfinite(values2)
    valid4 = np.isfinite(values4)
    if MIN_CASES is not None:
        valid2 &= counts2 >= int(MIN_CASES)
        valid4 &= counts4 >= int(MIN_CASES)

    masked2 = np.where(valid2, values2, np.nan)
    masked4 = np.where(valid4, values4, np.nan)
    absolute_diff = masked4 - masked2

    valid_relative = valid2 & valid4 & (np.abs(masked2) > RELATIVE_DIFFERENCE_EPS)
    relative_diff = np.full_like(absolute_diff, np.nan, dtype=float)
    relative_diff[valid_relative] = 100.0 * absolute_diff[valid_relative] / np.abs(masked2[valid_relative])

    vmin, vmax = symmetric_limits(masked2, masked4, fallback=metric_limits)
    avmin, avmax = symmetric_limits(absolute_diff)
    rvmin, rvmax = symmetric_limits(relative_diff)

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(0.50 * len(marker_names) * 4 + 3.4, 0.58 * len(marker_names) + 2.3),
        constrained_layout=True,
    )

    image0 = draw_heatmap(
        axes[0],
        masked2,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        low_count_mask=None if MIN_CASES is None else counts2 < int(MIN_CASES),
    )
    format_heatmap_axis(axes[0], title=DEPTH2_LABEL, show_ylabel=True)

    image1 = draw_heatmap(
        axes[1],
        masked4,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        low_count_mask=None if MIN_CASES is None else counts4 < int(MIN_CASES),
    )
    format_heatmap_axis(axes[1], title=DEPTH4_LABEL, show_ylabel=False)

    image2 = draw_heatmap(
        axes[2],
        absolute_diff,
        vmin=avmin,
        vmax=avmax,
        cmap=cmap,
        low_count_mask=~(valid2 & valid4),
    )
    format_heatmap_axis(axes[2], title=f"absolute diff\n{DIFFERENCE_LABEL}", show_ylabel=False)

    image3 = draw_heatmap(
        axes[3],
        relative_diff,
        vmin=rvmin,
        vmax=rvmax,
        cmap=cmap,
        low_count_mask=~valid_relative,
    )
    format_heatmap_axis(axes[3], title=f"relative diff (%)\n{DIFFERENCE_LABEL}", show_ylabel=False)

    cbar = fig.colorbar(image1, ax=axes[:2], fraction=0.046, pad=0.04)
    cbar.set_label(metric_spec["label"], fontsize=11)
    if metric_ticks is not None:
        cbar.set_ticks(metric_ticks)
    cbar.ax.tick_params(labelsize=9, length=4, direction="out")

    abs_cbar = fig.colorbar(image2, ax=axes[2], fraction=0.046, pad=0.04)
    abs_cbar.set_label(f"absolute difference in {metric_spec['label']}", fontsize=11)
    abs_cbar.ax.tick_params(labelsize=9, length=4, direction="out")

    rel_cbar = fig.colorbar(image3, ax=axes[3], fraction=0.046, pad=0.04)
    rel_cbar.set_label("relative difference (%)", fontsize=11)
    rel_cbar.ax.tick_params(labelsize=9, length=4, direction="out")

    fig.suptitle(rf"Center-resolved ablation comparison, $d_{{pert}} = {hop}$", fontsize=14, fontweight="bold")
    save_figure(fig, f"ablation_comparison_{metric_spec['short']}_dpert{hop}")
    plt.show()


**Population-Level Ablation Curves**

Overlay population-level ablation curves for the two runs. These are loaded from `population_effect_summary.csv` and scaled with the same metric convention as the center-resolved heatmaps.


In [ ]:
def scaled_population_table(run, metric):
    table = run["population"].copy()
    table = table[table["metric"] == metric].copy()
    if table.empty:
        raise ValueError(f"No population rows for metric {metric!r} in {run['run_dir']}")
    scale = METRIC_SPECS[metric]["scale"]
    for column in (
        "conditional_effect",
        "conditional_std",
        "conditional_sem",
        "prevalence_weighted_effect",
        "prevalence_weighted_std",
        "prevalence_weighted_sem",
    ):
        if column in table.columns:
            table[column] = pd.to_numeric(table[column], errors="coerce") * scale
    return table


population_tables = []
for label, run in runs.items():
    table = scaled_population_table(run, METRIC)
    table.insert(0, "run_label", label)
    table.insert(1, "run_dir", str(run["run_dir"]))
    population_tables.append(table)
combined_population_df = pd.concat(population_tables, ignore_index=True, sort=False)
combined_population_df.to_csv(TABLES_DIR / f"combined_{METRIC}_population_effect_summary.csv", index=False)


def plot_population_overlay(estimand_key, estimand_spec):
    value_column = estimand_spec["value"]
    sem_column = estimand_spec["sem"]
    n_markers = len(marker_names)
    n_cols = int(POPULATION_N_COLS)
    n_rows = int(np.ceil(n_markers / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.4 * n_cols, 2.55 * n_rows),
        sharex=True,
        sharey=True,
        squeeze=False,
        constrained_layout=True,
    )
    axes = axes.ravel()
    handles = []
    labels = []

    for ax, marker in zip(axes, marker_names):
        for run_label in (DEPTH2_LABEL, DEPTH4_LABEL):
            marker_df = combined_population_df[
                (combined_population_df["run_label"] == run_label)
                & (combined_population_df["marker"] == marker)
            ].copy()
            if POPULATION_HOPS_TO_PLOT is not None:
                marker_df = marker_df[marker_df["hop"].isin(POPULATION_HOPS_TO_PLOT)]
            marker_df = marker_df.sort_values("hop")
            if marker_df.empty:
                continue
            x = marker_df["hop"].to_numpy(float)
            y = marker_df[value_column].to_numpy(float)
            sem = marker_df[sem_column].to_numpy(float)
            color = RUN_COLORS.get(run_label)
            line, = ax.plot(
                x,
                y,
                marker="o",
                linewidth=1.6,
                markersize=4,
                color=color,
                label=run_label,
            )
            valid_band = np.isfinite(x) & np.isfinite(y) & np.isfinite(sem)
            if np.any(valid_band):
                ax.fill_between(
                    x[valid_band],
                    y[valid_band] - sem[valid_band],
                    y[valid_band] + sem[valid_band],
                    color=color,
                    alpha=0.18,
                    linewidth=0,
                )
            if marker == marker_names[0]:
                handles.append(line)
                labels.append(run_label)
        ax.axhline(0, color="0.25", linestyle="--", linewidth=0.8)
        ax.set_title(marker, fontsize=10, fontweight="bold")
        ax.grid(axis="y", alpha=0.22)

    for ax in axes[n_markers:]:
        ax.set_visible(False)
    for ax in axes[-n_cols:]:
        if ax.get_visible():
            ax.set_xlabel("distance from center (hop)")
    for row in range(n_rows):
        axes[row * n_cols].set_ylabel(metric_spec["label"])

    legend_ax = axes[n_markers] if n_markers < len(axes) else axes[-1]
    if not legend_ax.get_visible():
        legend_ax.set_visible(True)
        legend_ax.set_axis_off()
    legend_ax.legend(handles, labels, loc="lower left", frameon=False)

    fig.suptitle(
        f"Population ablation: {estimand_spec['label']}\n{metric_spec['label']} (shading: SEM)",
        fontsize=13,
        fontweight="bold",
    )
    save_figure(fig, f"population_overlay_{estimand_key}_{metric_spec['short']}")
    plt.show()
    return fig, axes


for estimand_key, estimand_spec in POPULATION_ESTIMANDS.items():
    plot_population_overlay(estimand_key, estimand_spec)


**Observed vs Predicted Variance Heatmaps**

If the loaded runs contain the variance summary tables produced by the updated `ablation_analysis.ipynb`, plot the empirical variance of center-cell curvature and the model-predicted variance for each marker-pair combination and perturbation distance.


In [ ]:
VARIANCE_VALUE_COLUMNS = {
    "true_curvature_variance": {
        "label": rf"observed curvature variance ({CURVATURE_SQUARED_UNIT_LABEL})",
        "scale": CURVATURE_SQUARED_TO_PHYSICAL_FACTOR,
        "short": "observed_variance",
    },
    "predicted_variance_original_mean": {
        "label": rf"mean predicted variance ({CURVATURE_SQUARED_UNIT_LABEL})",
        "scale": CURVATURE_SQUARED_TO_PHYSICAL_FACTOR,
        "short": "predicted_variance",
    },
}

variance_runs = {
    label: run
    for label, run in runs.items()
    if run.get("variance") is not None
}
missing_variance_runs = {
    label: run
    for label, run in runs.items()
    if run.get("variance") is None
}


def scaled_variance_table(run):
    table = run.get("variance")
    if table is None:
        return None
    table = table.copy()
    for column, spec in VARIANCE_VALUE_COLUMNS.items():
        if column in table.columns:
            table[column] = pd.to_numeric(table[column], errors="coerce") * spec["scale"]
    return table


def variance_heatmap_array(run, value_column, hop):
    table = scaled_variance_table(run)
    if table is None:
        raise ValueError(f"Run {run['run_dir']} has no variance summary table.")
    if value_column not in table.columns:
        raise ValueError(f"Variance table {run['variance_table_name']} has no column {value_column!r}.")
    if "table_label" in table.columns and table["table_label"].nunique(dropna=True) > 1:
        # Prefer the post-MAD summary if the combined table is loaded.
        preferred = table[table["table_label"].eq("post_non_overlap_post_mad")]
        if not preferred.empty:
            table = preferred
    subset = table[table["hop"].astype(int) == int(hop)]
    n = len(marker_names)
    values = np.full((n, n), np.nan, dtype=float)
    counts = np.zeros((n, n), dtype=int)
    for row in subset.itertuples(index=False):
        center_index = marker_to_index.get(row.center_marker)
        source_index = marker_value_to_index(row.source_marker)
        if center_index is None or source_index is None:
            continue
        values[center_index, source_index] = float(getattr(row, value_column))
        counts[center_index, source_index] = int(getattr(row, "cases", 0))
    return values, counts


if missing_variance_runs:
    print(
        "Variance summary tables missing for:\n- "
        + "\n- ".join(f"{label}: {run['run_dir']}" for label, run in missing_variance_runs.items())
    )

if not variance_runs:
    print("Skipping observed-vs-predicted variance heatmaps: no loaded run contains the required variance summary table.")
else:
    combined_variance_tables = []
    for label, run in variance_runs.items():
        table = scaled_variance_table(run)
        table.insert(0, "run_label", label)
        table.insert(1, "run_dir", str(run["run_dir"]))
        table.insert(2, "variance_table_name", run["variance_table_name"])
        combined_variance_tables.append(table)
    combined_variance_df = pd.concat(combined_variance_tables, ignore_index=True, sort=False)
    combined_variance_df.to_csv(TABLES_DIR / "combined_center_marker_pair_curvature_variance_summary.csv", index=False)

    for hop in HOPS_TO_PLOT:
        fig, axes = plt.subplots(
            len(VARIANCE_VALUE_COLUMNS),
            len(variance_runs),
            figsize=(0.50 * len(marker_names) * len(variance_runs) + 2.8, 0.55 * len(marker_names) * len(VARIANCE_VALUE_COLUMNS) + 2.4),
            squeeze=False,
            constrained_layout=True,
        )
        for row_index, (value_column, spec) in enumerate(VARIANCE_VALUE_COLUMNS.items()):
            arrays = []
            counts_by_label = {}
            for run_label, run in variance_runs.items():
                values, counts = variance_heatmap_array(run, value_column, hop)
                arrays.append(values)
                counts_by_label[run_label] = counts
            finite_parts = [arr[np.isfinite(arr)].ravel() for arr in arrays]
            finite = np.concatenate(finite_parts) if finite_parts else np.asarray([], dtype=float)
            if finite.size:
                vmin = 0.0
                vmax = float(np.nanpercentile(finite, 98))
                if not np.isfinite(vmax) or vmax <= 0:
                    vmax = float(np.nanmax(finite))
                if not np.isfinite(vmax) or vmax <= 0:
                    vmax = 1.0
            else:
                vmin, vmax = 0.0, 1.0
            image = None
            variance_cmap = plt.get_cmap("viridis").copy()
            variance_cmap.set_bad(BAD_COLOR)
            for col_index, (run_label, run) in enumerate(variance_runs.items()):
                values = arrays[col_index]
                counts = counts_by_label[run_label]
                valid = np.isfinite(values)
                if MIN_CASES is not None:
                    valid &= counts >= int(MIN_CASES)
                masked = np.where(valid, values, np.nan)
                image = draw_heatmap(
                    axes[row_index, col_index],
                    masked,
                    vmin=vmin,
                    vmax=vmax,
                    cmap=variance_cmap,
                    low_count_mask=None if MIN_CASES is None else counts < int(MIN_CASES),
                )
                format_heatmap_axis(
                    axes[row_index, col_index],
                    title=run_label if row_index == 0 else "",
                    show_ylabel=col_index == 0,
                )
                if col_index == 0:
                    axes[row_index, col_index].set_ylabel(f"{spec['label']}\ncenter marker")
            cbar = fig.colorbar(image, ax=list(axes[row_index, :]), fraction=0.046, pad=0.04)
            cbar.set_label(spec["label"], fontsize=10)
            cbar.ax.tick_params(labelsize=9, length=4, direction="out")

        fig.suptitle(rf"Observed vs predicted variance by marker pair, $d_{{pert}} = {hop}$", fontsize=14, fontweight="bold")
        save_figure(fig, f"marker_pair_observed_predicted_variance_dpert{hop}")
        plt.show()


**Save Settings**

In [ ]:
settings = {
    "created_at": datetime.now().isoformat(),
    "notebook": "experiments/ablation_tests.ipynb",
    "depth2_run_dir": str(runs[DEPTH2_LABEL]["run_dir"]),
    "depth4_run_dir": str(runs[DEPTH4_LABEL]["run_dir"]),
    "metric": METRIC,
    "hops_to_plot": HOPS_TO_PLOT,
    "min_cases": MIN_CASES,
    "relative_difference_definition": "100 * (depth4 - depth2) / abs(depth2)",
    "relative_difference_eps": RELATIVE_DIFFERENCE_EPS,
    "population_estimands": POPULATION_ESTIMANDS,
    "variance_tables": {
        label: run.get("variance_table_name")
        for label, run in runs.items()
    },
    "metric_scale": METRIC_SPECS[METRIC]["scale"],
    "metric_label": METRIC_SPECS[METRIC]["label"],
    "difference": f"{DEPTH4_LABEL} minus {DEPTH2_LABEL} after metric scaling",
    "marker_names": marker_names,
    "save_dir": str(SAVE_DIR),
}
with (SAVE_DIR / "settings.json").open("w") as handle:
    json.dump(settings, handle, indent=2)

print(f"Saved figures, tables, and settings to {SAVE_DIR}")
